In [1]:
import pandas as pd
import numpy as np
from rdkit.Chem import Descriptors, Draw
from rdkit import Chem
import json

In [2]:
allowed_descriptors = pd.read_json('all_peptide_descriptors.json')
allowed_descriptors = allowed_descriptors.iloc[:, 0].tolist()
allowed_descriptors

['peoe_vsa2',
 'ringcount',
 'kappa3',
 'fr_nh2',
 'estate_vsa10',
 'slogp_vsa3',
 'numheteroatoms',
 'nhohcount',
 'numhdonors',
 'peoe_vsa14',
 'numhacceptors',
 'bcut2d_logplow',
 'minabsestateindex',
 'peoe_vsa6',
 'bcut2d_chglo',
 'hallkieralpha',
 'smr_vsa1',
 'smr_vsa7',
 'chi3n',
 'minabspartialcharge',
 'numsaturatedheterocycles',
 'fpdensitymorgan3',
 'estate_vsa2',
 'kappa1',
 'fr_guanido',
 'nocount',
 'estate_vsa8',
 'minpartialcharge',
 'bcut2d_mrlow',
 'fr_unbrch_alkane',
 'chi0v',
 'slogp_vsa5',
 'estate_vsa7',
 'vsa_estate5',
 'estate_vsa11',
 'qed',
 'smr_vsa6',
 'numaromaticcarbocycles',
 'maxabspartialcharge',
 'minestateindex',
 'estate_vsa9',
 'estate_vsa3',
 'molmr',
 'chi0',
 'estate_vsa6',
 'fpdensitymorgan2',
 'smr_vsa4',
 'bertzct',
 'maxpartialcharge',
 'vsa_estate2',
 'vsa_estate8',
 'bcut2d_logphi',
 'numvalenceelectrons',
 'estate_vsa4',
 'estate_vsa5',
 'heavyatommolwt',
 'fr_benzene',
 'peoe_vsa9',
 'vsa_estate4',
 'slogp_vsa4',
 'slogp_vsa2',
 'fr_c_o_

## Nodes: Format Data & Get Features

In [3]:
peptide_nodes = pd.read_csv("../tables/SMILES_peptides_monomer.txt")
peptide_nodes['class'] = peptide_nodes.apply(lambda row: 'peptide_amd' if '_amd' in row['Molecule'] else 'peptide', axis=1)
peptide_nodes['type'] = 'node'

peptide_edges = pd.read_csv("../tables/SMILES_peptides_bond.txt")
peptide_edges['class'] = 'peptide'
peptide_edges['type'] = 'edge'

polymer_nodes = pd.read_csv("../tables_poly/SMILES_polymers_monomer.txt")
polymer_nodes['class'] = 'polymer'
polymer_nodes['type'] = 'node'

polymer_edges = pd.read_csv("../tables_poly/SMILES_polymers_bond.txt")
polymer_edges['class'] = 'polymer'
polymer_edges['type'] = 'edge'

df = pd.concat([peptide_nodes, polymer_nodes, peptide_edges, polymer_edges], ignore_index = True)
df = df.rename(columns = {'Molecule': 'monomer'})
df = df.drop(['Description'], axis=1)

In [4]:
df

,monomer,SMILES,class,type
0,G,C(C(=O)O)N,peptide,node
1,A,C[C@@H](C(=O)O)N,peptide,node
2,L,CC(C)C[C@@H](C(=O)O)N,peptide,node
3,M,CSCC[C@@H](C(=O)O)N,peptide,node
4,F,C1=CC=C(C=C1)C[C@@H](C(=O)O)N,peptide,node
5,W,C1=CC=C2C(=C1)C(=CN2)C[C@@H](C(=O)O)N,peptide,node
6,K,C(CCN)C[C@@H](C(=O)O)N,peptide,node
7,Q,C(CC(=O)N)[C@@H](C(=O)O)N,peptide,node
8,E,C(CC(=O)O)[C@@H](C(=O)O)N,peptide,node
9,S,C([C@@H](C(=O)O)N)O,peptide,node


In [5]:
%%capture
df['descriptors'] = df['SMILES'].apply(
    lambda x: Descriptors.CalcMolDescriptors(
        Chem.MolFromSmiles(x), missingVal=-9999, silent=True
    )
)

df['descriptors'] = df['descriptors'].apply(
    lambda descriptor_dict: {
        key: value for key, value in descriptor_dict.items() 
        if key.lower() in allowed_descriptors
    }
)

expanded_df = pd.json_normalize(df['descriptors'])
expanded_df.index = df.index
df = df.drop(columns=['descriptors'])
df = pd.concat([df, expanded_df], axis=1)

In [6]:
df

,monomer,SMILES,class,type,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,MolWt,...,fr_C_O_noCOO,fr_NH0,fr_NH1,fr_NH2,fr_amide,fr_benzene,fr_guanido,fr_imidazole,fr_para_hydroxylation,fr_unbrch_alkane
0,G,C(C(=O)O)N,peptide,node,9.243056,9.243056,0.277778,-0.967593,0.421171,75.067,...,0,0,0,1,0,0,0,0,0,0
1,A,C[C@@H](C(=O)O)N,peptide,node,9.574074,9.574074,0.731481,-0.962963,0.451352,89.094,...,0,0,0,1,0,0,0,0,0,0
2,L,CC(C)C[C@@H](C(=O)O)N,peptide,node,10.109769,10.109769,0.357500,-0.913241,0.583947,131.175,...,0,0,0,1,0,0,0,0,0,0
3,M,CSCC[C@@H](C(=O)O)N,peptide,node,10.070883,10.070883,0.552083,-0.912917,0.596996,149.215,...,0,0,0,1,0,0,0,0,0,0
4,F,C1=CC=C(C=C1)C[C@@H](C(=O)O)N,peptide,node,10.378642,10.378642,0.385093,-0.959395,0.690463,165.192,...,0,0,0,1,0,1,0,0,0,0
5,W,C1=CC=C2C(=C1)C(=CN2)C[C@@H](C(=O)O)N,peptide,node,10.626394,10.626394,0.346574,-0.971962,0.700584,204.229,...,0,0,1,1,0,1,0,0,1,0
6,K,C(CCN)C[C@@H](C(=O)O)N,peptide,node,10.137222,10.137222,0.520370,-0.933313,0.457177,146.190,...,0,0,0,2,0,0,0,0,0,1
7,Q,C(CC(=O)N)[C@@H](C(=O)O)N,peptide,node,10.100084,10.100084,0.021296,-1.109954,0.456092,146.146,...,1,0,0,2,1,0,0,0,0,0
8,E,C(CC(=O)O)[C@@H](C(=O)O)N,peptide,node,9.993880,9.993880,0.023148,-1.165509,0.485976,147.130,...,0,0,0,1,0,0,0,0,0,0
9,S,C([C@@H](C(=O)O)N)O,peptide,node,9.645324,9.645324,0.504630,-1.178241,0.394240,105.093,...,0,0,0,1,0,0,0,0,0,0


## Analyze Features

### Ensure each class has >1 

In [7]:
def get_unique(X):
    """
        returns: unique descriptors, homogenous descriptors
    """
    X = X.apply(set, axis=0).map(len)
    return set(X[X>1].index.tolist())

In [8]:
descriptors = df.select_dtypes(include='number').columns.tolist()

In [9]:
# Verify
# set([desc.lower() for desc in descriptors]) == set(allowed_descriptors)

### Node Descriptors:

In [10]:
unique_descriptors_peptide = get_unique(df[(df['type'] == 'node') & (df['class'] == 'peptide')][descriptors])
unique_descriptors_peptide_amd = get_unique(df[(df['type'] == 'node') & (df['class'] == 'peptide_amd')][descriptors])
unique_descriptors_polymer = get_unique(df[(df['type'] == 'node') & (df['class'] == 'polymer')][descriptors])
node_descriptors = list(unique_descriptors_peptide & unique_descriptors_peptide_amd & unique_descriptors_polymer)

### Edge Descriptors:

In [11]:
edge_descriptors = list(get_unique(df[df['type'] == 'edge'][descriptors]))

## Export to .json

In [12]:
df[df['type'] == 'node'][sum([['monomer', 'SMILES', 'class', 'type'], node_descriptors],[])].to_csv('node_feats.csv', index=False)
df[df['type'] == 'edge'][sum([['monomer', 'SMILES', 'class', 'type'], edge_descriptors],[])].to_csv('edge_feats.csv', index=False)

In [13]:
json_data = pd.DataFrame({'node': [node_descriptors], 'edge': [edge_descriptors]})

In [14]:
json_data.to_json('../unique_descriptors.json', orient='records')

In [15]:
pd.read_json('../unique_descriptors.json').to_dict(orient='records')[0]

{'node': ['qed',
  'NumHeteroatoms',
  'EState_VSA5',
  'BCUT2D_CHGHI',
  'VSA_EState6',
  'Chi4n',
  'VSA_EState5',
  'FpDensityMorgan3',
  'Chi1v',
  'HallKierAlpha',
  'NumRotatableBonds',
  'PEOE_VSA7',
  'EState_VSA4',
  'BCUT2D_CHGLO',
  'Chi1n',
  'fr_unbrch_alkane',
  'EState_VSA9',
  'NumHAcceptors',
  'NumSaturatedHeterocycles',
  'HeavyAtomCount',
  'VSA_EState7',
  'SMR_VSA6',
  'SlogP_VSA4',
  'fr_NH0',
  'MinPartialCharge',
  'NumHDonors',
  'EState_VSA3',
  'Chi3v',
  'TPSA',
  'SMR_VSA7',
  'MaxPartialCharge',
  'PEOE_VSA9',
  'BertzCT',
  'fr_guanido',
  'Chi4v',
  'MaxAbsPartialCharge',
  'MolLogP',
  'SMR_VSA4',
  'PEOE_VSA1',
  'Kappa2',
  'MinEStateIndex',
  'VSA_EState8',
  'Chi0n',
  'SMR_VSA5',
  'fr_para_hydroxylation',
  'EState_VSA2',
  'LabuteASA',
  'ExactMolWt',
  'NumAromaticRings',
  'Chi1',
  'SlogP_VSA1',
  'BCUT2D_LOGPHI',
  'fr_NH1',
  'SMR_VSA1',
  'Chi3n',
  'MaxAbsEStateIndex',
  'SMR_VSA10',
  'EState_VSA6',
  'NOCount',
  'MolWt',
  'Chi2n',
  '

## Scaling / Normalization:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import QuantileTransformer

In [ ]:
original_data = new_df['vsa_estate3']

# Box-Cox
# scaled_data = stats.zscore(original_data)
# scaled_data = scaled_data + abs(scaled_data.min()) + 0.01
# normalized_data = stats.boxcox(scaled_data)

# Yeo-Johnson
# scaled_data = stats.zscore(original_data)
# normalized_data = stats.yeojohnson(scaled_data)

# Quantile Transformer
# quantile = QuantileTransformer(output_distribution='normal', n_quantiles = 52)
# normalized_data = quantile.fit_transform(np.array(original_data).reshape(-1, 1))

In [ ]:
original_data = new_df['chi1']
quantile = QuantileTransformer(output_distribution='normal', n_quantiles = 52)
normalized_data = quantile.fit_transform(np.array(original_data).reshape(-1, 1))

fig, ax=plt.subplots(1, 3, figsize=(15, 3))
sns.histplot(original_data, ax=ax[0], kde=True, legend=False)
ax[0].set_title("Original Data")
sns.histplot(scaled_data, ax=ax[1], kde=True, legend=False)
ax[1].set_title("Scaled Data")
sns.histplot(normalized_data, ax=ax[2], kde=True, legend=False)
ax[2].set_title("Normalized data")
plt.show()

In [ ]:
from scipy.stats import shapiro
shapiro(normalized_data)

In [ ]:
# Of the descriptors, compute the nonzero percent for classes 'peptide', 'polymer'; 
# divide min / max and see if > threshold %